## 주문(orders)

In [ ]:
from supabase_client import get_supabase_client

supabase = get_supabase_client()

In [ ]:
from datetime import datetime, timezone

class order_service:
    def __init__(self, supabase):
        session = supabase.auth.get_user()
        if not session or not session.user:
            raise PermissionError("로그인이 필요합니다.")
        self.supabase = supabase
        self.user = session.user

    def get_orders(self):
        return (self.supabase.table("orders").select("*")
                .eq("user_id", self.user.id).is_("deleted_at", "null").execute().data)

    def get_order_info(self, order_id):
        return (self.supabase.table("orders").select("*")
                .eq("id", order_id).eq("user_id", self.user.id)
                .is_("deleted_at", "null").execute().data)

    def add_order(
        self, *, order_no, order_name, order_email, order_phone,
        receiver_name, receiver_phone, zipcode, address, address_sub, total_price
    ):
        return self.supabase.table("orders").insert({
            "order_no": order_no,
            "order_name": order_name,
            "order_email": order_email,
            "order_phone": order_phone,
            "receiver_name": receiver_name,
            "receiver_phone": receiver_phone,
            "zipcode": zipcode,
            "address": address,
            "address_sub": address_sub,
            "total_price": total_price,
            "user_id": self.user.id,
        }).execute().data

    def set_order(self, order_id, *, address=None, address_sub=None, total_price=None):
        changes = {
            key: value for key, value in {
                "address": address, "address_sub": address_sub, "total_price": total_price
            }.items() if value is not None
        }
        if not changes:
            return []
        changes["modified_at"] = datetime.now(timezone.utc).isoformat()
        return (self.supabase.table("orders").update(changes)
                .eq("id", order_id).eq("user_id", self.user.id)
                .is_("deleted_at", "null").execute().data)

    def remove_order(self, order_id):
        return (self.supabase.table("orders")
                .update({"deleted_at": datetime.now(timezone.utc).isoformat()})
                .eq("id", order_id).eq("user_id", self.user.id)
                .is_("deleted_at", "null").execute().data)
